# Fabric metadata-driven transformation engine

This full-load notebook reads an Excel mapping workbook from a Fabric Lakehouse, loads an array of complete Bronze or Silver source datasets, applies configured source filters, joins and target-column expressions, and overwrites one managed Delta target table.

Pipeline parameters are defined in the next cell. The Excel workbook must retain these sheets: Control, Sources, Mappings, Joins and Filters.

Operational notes:

- Attach the Lakehouse that contains the mapping workbook as the notebook's default Lakehouse.
- Add openpyxl to the Fabric Environment used by this notebook. Do not install it during every scheduled run.
- Every source path must identify the complete dataset required for the full load.
- The notebook creates or updates a managed Delta table. Delta controls the physical Parquet filenames.

In [ ]:
# Pipeline parameters
source_tables_json = '["orders", "customers", "products"]'
target_table = "silver.sales_order_enriched"
mapping_file_path = "Files/framework/mappings/sales/sales_order_enriched_mapping.xlsx"

# yyyy-MM-dd. Blank means the current UTC date.
logical_date = ""

pipeline_run_id = ""

# Optional JSON object keyed by SourceTable. Overrides the path selected from the workbook.
# Example: {"orders":"Files/bronze/sales/2026/09/24/orders_*.parquet"}
source_path_overrides_json = "{}"

# Only validate and generate SQL. Do not write the target.
dry_run = False

In [ ]:
import json
import os
import re
import tempfile
from datetime import datetime, timezone

import pandas as pd
from pyspark.sql import functions as F

IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
TARGET_IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*(?:\.[A-Za-z_][A-Za-z0-9_]*)?$")
TARGET_TYPE = re.compile(
    r"^(STRING|BOOLEAN|BYTE|TINYINT|SHORT|SMALLINT|INT|INTEGER|LONG|BIGINT|"
    r"FLOAT|DOUBLE|DATE|TIMESTAMP|BINARY|DECIMAL\([1-9][0-9]?,[0-9]+\))$",
    flags=re.IGNORECASE,
)
PROHIBITED_SQL = re.compile(
    r"\b(DROP|TRUNCATE|ALTER|CREATE|INSERT|UPDATE|DELETE|MERGE|GRANT|REVOKE)\b",
    flags=re.IGNORECASE,
)


def as_bool(value, default=False):
    if pd.isna(value):
        return default
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        return value != 0
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def text_value(value, default=""):
    if pd.isna(value):
        return default
    return str(value).strip()


def validate_identifier(value, label):
    if not IDENTIFIER.fullmatch(value):
        raise ValueError(f"Invalid {label}: {value}")


def validate_expression(expression, label):
    expression = text_value(expression)
    if not expression:
        raise ValueError(f"{label} cannot be blank")
    if ";" in expression or "--" in expression or "/*" in expression:
        raise ValueError(f"{label} contains a prohibited SQL separator or comment")
    match = PROHIBITED_SQL.search(expression)
    if match:
        raise ValueError(f"{label} contains prohibited keyword: {match.group(1)}")
    return expression


def resolve_mapping_file(path):
    path = path.strip()
    if path.startswith("Files/"):
        return f"/lakehouse/default/{path}"
    if path.startswith("/lakehouse/") or path.startswith("/tmp/"):
        return path
    if path.startswith("abfss://"):
        local_path = os.path.join(tempfile.gettempdir(), os.path.basename(path))
        notebookutils.fs.cp(path, f"file:{local_path}")
        return local_path
    raise ValueError(
        "mapping_file_path must use Files/..., /lakehouse/..., /tmp/... or an abfss:// path"
    )


def clean_sheet(frame):
    frame = frame.dropna(how="all").copy()
    frame.columns = [str(c).strip() for c in frame.columns]
    return frame


def load_mapping_workbook(path):
    local_path = resolve_mapping_file(path)
    try:
        sheets = pd.read_excel(local_path, sheet_name=None, header=3, engine="openpyxl")
    except ImportError as exc:
        raise RuntimeError(
            "openpyxl is required. Add openpyxl to the Fabric Environment attached to this notebook."
        ) from exc

    required = {"Control", "Sources", "Mappings", "Joins", "Filters"}
    missing = required.difference(sheets)
    if missing:
        raise ValueError(f"Mapping workbook is missing sheets: {sorted(missing)}")
    return {name: clean_sheet(sheets[name]) for name in required}


def require_columns(frame, sheet_name, columns):
    missing = set(columns).difference(frame.columns)
    if missing:
        raise ValueError(f"{sheet_name} is missing columns: {sorted(missing)}")

In [ ]:
# Load and validate the mapping workbook
sheets = load_mapping_workbook(mapping_file_path)
control_df = sheets["Control"]
sources_df = sheets["Sources"]
mappings_df = sheets["Mappings"]
joins_df = sheets["Joins"]
filters_df = sheets["Filters"]

require_columns(control_df, "Control", ["Parameter", "Value"])
require_columns(
    sources_df,
    "Sources",
    [
        "SourceOrder", "SourceAlias", "SourceTable", "SourcePathTemplate",
        "FileFormat", "IsBaseSource", "Active"
    ],
)
require_columns(
    mappings_df,
    "Mappings",
    [
        "MappingOrder", "SourceExpression", "TargetColumn", "TargetDataType",
        "Nullable", "DefaultExpression", "Active", "ExampleOnly"
    ],
)
require_columns(
    joins_df,
    "Joins",
    ["JoinOrder", "LeftAlias", "RightAlias", "JoinType", "JoinCondition", "Active"],
)
require_columns(
    filters_df,
    "Filters",
    ["FilterOrder", "Stage", "SourceAlias", "FilterCondition", "Active", "ExampleOnly"],
)

control = {
    text_value(row["Parameter"]): text_value(row["Value"])
    for _, row in control_df.iterrows()
    if text_value(row["Parameter"])
}

configured_target = control.get("TargetTable", "")
if not target_table:
    target_table = configured_target
elif configured_target and target_table.lower() != configured_target.lower():
    raise ValueError(
        f"target_table parameter '{target_table}' does not match Control.TargetTable "
        f"'{configured_target}'"
    )

if not TARGET_IDENTIFIER.fullmatch(target_table):
    raise ValueError(f"Invalid target_table: {target_table}")

write_strategy = control.get("WriteStrategy", "OVERWRITE").upper()
if write_strategy != "OVERWRITE":
    raise ValueError("This full-load notebook requires WriteStrategy=OVERWRITE")

schema_mode = control.get("SchemaMode", "STRICT").upper()
if schema_mode not in {"STRICT", "ADD_COLUMNS"}:
    raise ValueError(f"Unsupported SchemaMode: {schema_mode}")

partition_columns = [
    x.strip() for x in control.get("TargetPartitionColumns", "").split(",") if x.strip()
]

for column in partition_columns:
    validate_identifier(column, "TargetPartitionColumns entry")

requested_sources = json.loads(source_tables_json)
if not isinstance(requested_sources, list) or not all(isinstance(x, str) for x in requested_sources):
    raise ValueError("source_tables_json must be a JSON array of table-name strings")
requested_sources = [x.strip() for x in requested_sources if x.strip()]
if len(requested_sources) != len(set(x.lower() for x in requested_sources)):
    raise ValueError("source_tables_json contains duplicate table names")

path_overrides = json.loads(source_path_overrides_json or "{}")
if not isinstance(path_overrides, dict):
    raise ValueError("source_path_overrides_json must be a JSON object")

if logical_date:
    run_date = datetime.strptime(logical_date, "%Y-%m-%d").date()
else:
    run_date = datetime.now(timezone.utc).date()

sources_df = sources_df[sources_df["Active"].map(as_bool)].copy()
sources_df["SourceOrder"] = pd.to_numeric(sources_df["SourceOrder"], errors="raise")
sources_df = sources_df.sort_values("SourceOrder")

configured_sources = [text_value(x) for x in sources_df["SourceTable"]]
if {x.lower() for x in configured_sources} != {x.lower() for x in requested_sources}:
    raise ValueError(
        "source_tables_json must match all active SourceTable values in the workbook. "
        f"Parameter={requested_sources}; workbook={configured_sources}"
    )

base_sources = sources_df[sources_df["IsBaseSource"].map(as_bool)]
if len(base_sources) != 1:
    raise ValueError("Exactly one active source must have IsBaseSource=TRUE")

print(
    json.dumps(
        {
            "target_table": target_table,
            "load_type": "FULL",
            "write_strategy": write_strategy,
            "logical_date": run_date.isoformat(),
            "source_tables": requested_sources,
            "mapping_file": mapping_file_path,
        },
        indent=2,
    )
)

In [ ]:
# Load source datasets and register logical aliases
def render_path(template, source_table):
    replacements = {
        "{table}": source_table,
        "{yyyy}": run_date.strftime("%Y"),
        "{MM}": run_date.strftime("%m"),
        "{dd}": run_date.strftime("%d"),
        "{logical_date}": run_date.isoformat(),
    }
    rendered = template
    for token, value in replacements.items():
        rendered = rendered.replace(token, value)
    return rendered


def resolve_source_path(path):
    if path.startswith("Files/") or path.startswith("Tables/"):
        return f"/lakehouse/default/{path}"
    return path


def read_source(row):
    source_table = text_value(row["SourceTable"])
    source_alias = text_value(row["SourceAlias"])
    file_format = text_value(row["FileFormat"]).lower()

    validate_identifier(source_alias, "SourceAlias")
    validate_identifier(source_table, "SourceTable")
    if file_format not in {"parquet", "delta", "csv"}:
        raise ValueError(f"Unsupported FileFormat for {source_table}: {file_format}")
    if source_table in path_overrides:
        selected_path = text_value(path_overrides[source_table])
    else:
        selected_path = text_value(row["SourcePathTemplate"])

    if not selected_path:
        raise ValueError(f"No source path configured for {source_table}")

    rendered_path = resolve_source_path(render_path(selected_path, source_table))
    reader = spark.read.format(file_format)
    if file_format == "csv":
        reader = reader.option("header", "true").option("inferSchema", "false")

    print(f"Reading {source_table} as {source_alias} from {rendered_path}")
    return source_alias, reader.load(rendered_path)


active_filters = filters_df[
    filters_df["Active"].map(as_bool) & ~filters_df["ExampleOnly"].map(as_bool)
].copy()
active_filters["FilterOrder"] = pd.to_numeric(active_filters["FilterOrder"], errors="raise")
active_filters = active_filters.sort_values("FilterOrder")

loaded_sources = {}
for _, source_row in sources_df.iterrows():
    alias, source_df = read_source(source_row)
    raw_view = f"_raw_{alias}"
    source_df.createOrReplaceTempView(raw_view)

    source_conditions = active_filters[
        (active_filters["Stage"].astype(str).str.upper() == "SOURCE")
        & (active_filters["SourceAlias"].astype(str).str.strip() == alias)
    ]

    if len(source_conditions):
        predicates = [
            validate_expression(row["FilterCondition"], f"source filter {row['FilterOrder']}")
            for _, row in source_conditions.iterrows()
        ]
        filtered_df = spark.sql(
            f"SELECT * FROM `{raw_view}` {alias} WHERE " + " AND ".join(f"({p})" for p in predicates)
        )
    else:
        filtered_df = source_df

    filtered_df.createOrReplaceTempView(alias)
    loaded_sources[alias] = filtered_df

print(f"Registered source aliases: {sorted(loaded_sources)}")

In [ ]:
# Generate one Spark SQL SELECT statement from the workbook
active_mappings = mappings_df[
    mappings_df["Active"].map(as_bool) & ~mappings_df["ExampleOnly"].map(as_bool)
].copy()
active_mappings["MappingOrder"] = pd.to_numeric(active_mappings["MappingOrder"], errors="raise")
active_mappings = active_mappings.sort_values("MappingOrder")

if active_mappings.empty:
    raise ValueError("No active target mappings were found")

select_items = []
target_columns = []
for _, row in active_mappings.iterrows():
    expression = validate_expression(row["SourceExpression"], f"mapping {row['MappingOrder']}")
    target_column = text_value(row["TargetColumn"])
    target_type = text_value(row["TargetDataType"]).upper()
    default_expression = text_value(row["DefaultExpression"])

    validate_identifier(target_column, "TargetColumn")
    if not target_type:
        raise ValueError(f"TargetDataType is blank for {target_column}")
    if not TARGET_TYPE.fullmatch(target_type):
        raise ValueError(f"Unsupported or invalid TargetDataType for {target_column}: {target_type}")
    if default_expression:
        default_expression = validate_expression(
            default_expression, f"default expression for {target_column}"
        )
        expression = f"COALESCE(({expression}), ({default_expression}))"

    select_items.append(f"CAST(({expression}) AS {target_type}) AS `{target_column}`")
    target_columns.append(target_column)

if len(target_columns) != len(set(x.lower() for x in target_columns)):
    raise ValueError("Mappings contains duplicate TargetColumn values")

base_alias = text_value(base_sources.iloc[0]["SourceAlias"])
from_clause = f"FROM `{base_alias}` {base_alias}"

active_joins = joins_df[joins_df["Active"].map(as_bool)].copy()
if not active_joins.empty:
    active_joins["JoinOrder"] = pd.to_numeric(active_joins["JoinOrder"], errors="raise")
    active_joins = active_joins.sort_values("JoinOrder")

join_clauses = []
joined_aliases = {base_alias}
for _, row in active_joins.iterrows():
    left_alias = text_value(row["LeftAlias"])
    right_alias = text_value(row["RightAlias"])
    join_type = text_value(row["JoinType"]).upper()
    condition = validate_expression(row["JoinCondition"], f"join {row['JoinOrder']}")

    if left_alias not in joined_aliases:
        raise ValueError(
            f"Join {row['JoinOrder']} references left alias '{left_alias}' before it is joined"
        )
    if right_alias not in loaded_sources:
        raise ValueError(f"Join references unknown right alias: {right_alias}")
    if right_alias in joined_aliases:
        raise ValueError(f"Alias is joined more than once: {right_alias}")
    if join_type not in {"INNER", "LEFT", "RIGHT", "FULL"}:
        raise ValueError(f"Unsupported JoinType: {join_type}")

    join_clauses.append(
        f"{join_type} JOIN `{right_alias}` {right_alias} ON {condition}"
    )
    joined_aliases.add(right_alias)

unused_aliases = set(loaded_sources).difference(joined_aliases)
if unused_aliases:
    raise ValueError(f"Active sources are not connected by the join rules: {sorted(unused_aliases)}")

final_filters = active_filters[active_filters["Stage"].astype(str).str.upper() == "FINAL"]
where_clause = ""
if len(final_filters):
    final_predicates = [
        validate_expression(row["FilterCondition"], f"final filter {row['FilterOrder']}")
        for _, row in final_filters.iterrows()
    ]
    where_clause = "WHERE " + " AND ".join(f"({p})" for p in final_predicates)

generated_sql = "\n".join(
    [
        "SELECT",
        "    " + ",\n    ".join(select_items),
        from_clause,
        *join_clauses,
        where_clause,
    ]
).strip()

print("Generated Spark SQL:\n")
print(generated_sql)

result_df = spark.sql(generated_sql)
result_df = (
    result_df
    .withColumn("_transformation_timestamp", F.current_timestamp())
    .withColumn("_pipeline_run_id", F.lit(pipeline_run_id))
    .withColumn("_logical_date", F.lit(run_date.isoformat()).cast("date"))
    .withColumn("_load_type", F.lit("FULL"))
    .withColumn("_mapping_file", F.lit(mapping_file_path))
)

for _, row in active_mappings.iterrows():
    column_name = text_value(row["TargetColumn"])
    nullable = as_bool(row["Nullable"], default=True)
    if not nullable and result_df.filter(F.col(column_name).isNull()).limit(1).count() > 0:
        raise ValueError(f"Non-nullable target column contains null values: {column_name}")

output_rows = result_df.count()
print(f"Rows produced: {output_rows}")
result_df.printSchema()
display(result_df.limit(20))

In [ ]:
# Write the managed Delta target
def target_exists(name):
    return spark.catalog.tableExists(name)


def create_target_schema_if_required(name):
    if "." in name:
        schema_name, _ = name.split(".", 1)
        validate_identifier(schema_name, "target schema")
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{schema_name}`")


def validate_existing_schema(incoming_df, name):
    if not target_exists(name) or schema_mode == "ADD_COLUMNS":
        return
    existing = {field.name.lower(): field.dataType.simpleString() for field in spark.table(name).schema.fields}
    incoming = {field.name.lower(): field.dataType.simpleString() for field in incoming_df.schema.fields}
    missing_in_target = sorted(set(incoming).difference(existing))
    missing_in_input = sorted(set(existing).difference(incoming))
    type_changes = sorted(
        key for key in set(existing).intersection(incoming) if existing[key] != incoming[key]
    )
    if missing_in_target or missing_in_input or type_changes:
        raise ValueError(
            "STRICT schema validation failed. "
            f"New columns={missing_in_target}; missing input columns={missing_in_input}; "
            f"type changes={type_changes}"
        )


if dry_run:
    print("dry_run=True. Target write skipped.")
else:
    create_target_schema_if_required(target_table)
    validate_existing_schema(result_df, target_table)

    writer = result_df.write.format("delta").mode("overwrite")
    if schema_mode == "ADD_COLUMNS":
        writer = writer.option("overwriteSchema", "true")
    if partition_columns and not target_exists(target_table):
        writer = writer.partitionBy(*partition_columns)
    writer.saveAsTable(target_table)

    print(f"Target write completed: {target_table}")

In [ ]:
# Return a compact status object to the calling Fabric pipeline
result = {
    "status": "SUCCEEDED",
    "target_table": target_table,
    "load_type": "FULL",
    "write_strategy": write_strategy,
    "logical_date": run_date.isoformat(),
    "source_count": len(requested_sources),
    "output_rows": output_rows,
    "pipeline_run_id": pipeline_run_id,
    "dry_run": bool(dry_run),
}

result_json = json.dumps(result)
print(result_json)
notebookutils.notebook.exit(result_json)